# Quick start: real data (ETTh1)
Forecast transformer oil temperature (`OT`) 24 hours ahead from the last 96 hours of all seven ETTh1 measurements.
Run with the **DynForecastLab (.venv)** kernel. The CSV is downloaded to `data/raw/` on first use. Short epochs keep this to a few minutes; they are an execution check, not a benchmark.

In [ ]:
from pathlib import Path
import os
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dynforecast.experiments import run_experiment

In [ ]:
def summary(runs):
    return pd.DataFrame([{"model": r["config"]["model"]["name"], "status": r["status"],
                          **{k: r.get("metrics", {}).get(k) for k in ("mae", "rmse", "mase")}}
                         for r in runs]).sort_values("rmse")

In [ ]:
def plot_forecast(run, target=0, window=0):
    with np.load(Path(run["config"]["output"]) / run["run_id"] / "predictions.npz") as a:
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(a["history_times"][window], a["history"][window, :, target], color="#9aa8ba", label="History")
        ax.plot(a["target_times"][window], a["truth"][window, :, target], color="#17263d", label="Observed future")
        ax.plot(a["target_times"][window], a["predictions"][window, :, target], color="#d1495b", label="Forecast")
        ax.set_title(run["config"]["model"]["name"]); ax.legend()

# Look at the data

In [ ]:
from dynforecast.experiments.runner import load_series
ett = load_series({"name": "ett", "max_rows": 3000})
print(ett.names, ett.observations.shape)
plt.figure(figsize=(10, 3)); plt.plot(ett.observations[:500, 6]); plt.title("OT, first 500 hours");

# Compare a few model families
Same data, split, history and horizon for every model; target index 6 is `OT`.

In [ ]:
base = {"dataset": {"name": "ett", "max_rows": 3000}, "targets": [6], "history": 96, "horizon": 24,
        "stride": 24, "output": "results/notebook_real"}
runs = [run_experiment({**base, "model": m}) for m in (
    {"name": "persistence"},
    {"name": "ridge"},
    {"name": "dlinear", "epochs": 5},
    {"name": "lstm", "epochs": 5, "width": 32},
)]
summary(runs)

In [ ]:
With 5 epochs on 3,000 rows, persistence usually wins on ETT: the oil temperature drifts and the neural models are barely trained. Raise `epochs` and `max_rows` (or run `configs/experiments/real_data.yaml`) for a real comparison.

In [ ]:
best = min((r for r in runs if r["status"] == "completed"), key=lambda r: r["metrics"]["rmse"])
plot_forecast(best)

# Next
- Full ETT study: `configs/experiments/real_data.yaml` (see the README for the CLI).
- Your own file: `04_your_own_csv.ipynb`.